# 05 Technology Mix

Deep dive into reactor technology families: what's operating, what's in the pipeline, and how the maturity scoring model distinguishes commercial technologies from early-stage concepts.

> Run `python run_pipeline.py` first.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
processed = ROOT / 'data' / 'processed'
predictions = ROOT / 'outputs' / 'predictions'

reactors = pd.read_csv(processed / 'reactors_master.csv')
pipeline = pd.read_csv(processed / 'reactor_pipeline.csv')
taxonomy = pd.read_csv(processed / 'technology_taxonomy.csv')
print(f"Technology families: {taxonomy.technology_family.nunique()}")
print(f"Reactor types: {len(taxonomy)}")
taxonomy[['technology_family','reactor_type','maturity_score','deployment_status','smr_flag','geniv_flag']].sort_values('maturity_score', ascending=False)

## Operating fleet vs pipeline — technology comparison

In [ ]:
op_tech = reactors[reactors.status_group=='Operating'].groupby('technology_family')['capacity_mwe'].sum().div(1000)
pl_tech = pipeline.groupby('technology_family')['capacity_mwe'].sum().div(1000)

compare = pd.DataFrame({'Operating (GWe)': op_tech, 'Pipeline (GWe)': pl_tech}).fillna(0).sort_values('Operating (GWe)', ascending=False)

ax = compare.plot.bar(figsize=(11, 5), colormap='Paired', width=0.75)
ax.set_title('Technology Mix: Operating Fleet vs Pipeline (GWe)', fontsize=13)
ax.set_ylabel('GWe'); ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()

print("Pipeline shift toward advanced technologies:")
print((compare['Pipeline (GWe)'] / compare['Pipeline (GWe)'].sum() * 100).round(1).to_string())

**Key insight:** The pipeline has a larger share of SMR and Gen IV technologies relative to the operating fleet. Whether this translates to commissioned capacity depends on financing, supply chain, and licensing timelines — which is why these are scored as *signals*, not forecasts.

## Technology maturity scoring

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
tax_sorted = taxonomy.sort_values('maturity_score', ascending=False)
colors = tax_sorted['technology_family'].map({
    'Light Water Reactor': '#2196F3', 'Heavy Water Reactor': '#03A9F4',
    'Small Modular Reactor': '#4CAF50', 'Fast Reactor': '#FF9800',
    'Gas-Cooled Reactor': '#9C27B0', 'Molten Salt Reactor': '#F44336',
    'Microreactor': '#795548', 'Thorium Fuel Cycle': '#607D8B',
})
bars = ax.barh(tax_sorted['reactor_type'], tax_sorted['maturity_score'], color=colors)
ax.set_xlabel('Maturity Score (0–100)')
ax.set_title('Technology Maturity Score by Reactor Type', fontsize=13)
ax.axvline(75, color='green', linestyle='--', alpha=0.6, label='Commercial (≥75)')
ax.axvline(55, color='orange', linestyle='--', alpha=0.6, label='Commercializing (≥55)')
ax.axvline(30, color='red', linestyle='--', alpha=0.6, label='Demonstration (≥30)')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## What goes into the maturity score

```
maturity_score = base_maturity
               + known_operating_units × 1.2
               + known_under_construction_units × 1.8
```

`base_maturity` encodes deployment status (commercial, commercializing, first deployment, demonstration, concept). Operating and construction unit counts from the sample pipeline adjust the score upward for technologies with real-world deployment evidence.

## SMR and Gen IV watch — maturity vs deployment reality

In [ ]:
adv = taxonomy[(taxonomy.smr_flag==True) | (taxonomy.geniv_flag==True)]

fig = px.scatter(
    adv,
    x='known_operating_units', y='maturity_score',
    size=adv['known_under_construction_units'].clip(lower=1),
    color='technology_family', hover_name='reactor_type',
    text='reactor_type',
    title='SMR / Gen IV: Maturity Score vs Real Deployment (bubble = under construction)',
    template='plotly_white',
    labels={'known_operating_units': 'Known operating units', 'maturity_score': 'Maturity score'}
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.show()

**Reading this chart:**
- Top-right: highest maturity AND operating units → commercial reality
- Top-left: high maturity score but no operating units → overconfident claims need scrutiny
- Bottom-right: units operating but low score → often older Gen II types not in scope
- Bottom-left: concepts and R&D → treat as long-horizon options, not near-term pipeline

This is the key analytical lens the platform applies: **deployment evidence, not vendor claims, drives the scores**.

## Generation category breakdown

In [ ]:
gen_cat = reactors.groupby('generation_category')['capacity_mwe'].agg(['sum','count']).reset_index()
gen_cat.columns = ['generation_category','total_mwe','units']
gen_cat['capacity_gwe'] = gen_cat['total_mwe'] / 1000

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].pie(gen_cat['capacity_gwe'], labels=gen_cat['generation_category'],
            autopct='%1.0f%%', startangle=140)
axes[0].set_title('Capacity by Generation Category')

sns.barplot(data=gen_cat, x='generation_category', y='units', ax=axes[1], palette='Blues_d')
axes[1].set_title('Unit Count by Generation Category')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout(); plt.show()

## Fuel cycle complexity

In [ ]:
fuel_info = taxonomy[['reactor_type','technology_family','fuel_cycle','coolant','moderator','neutron_spectrum','maturity_score']].sort_values('maturity_score', ascending=False)
fuel_info